
# WSD с WordNet/OMW и ConceptNet

**Формат:** единый Jupyter-ноутбук с готовым каркасом. Вам нужно реализовать несколько функций (помечены `TODO`), а затем запустить тесты (assert'ы).  

**Что внутри:**  
- Мини-набор предложений (EN/ES/IT).  
- Крошечный офлайн-файл `ConceptNet-mini` (встроен в ячейку).  
- Функции для: извлечения кандидатов из WordNet/OMW, простого Lesk, подсчётов кой-чего по ConceptNet, и др.
- Тесты с ожидаемыми числами

## Установка и загрузка корпусов NLTK

In [13]:
! pip install nltk

In [14]:

import nltk, sys, math, re, csv, io
from typing import List, Tuple, Optional, Dict
import itertools

# Скачаем требуемые корпуса (можно запускать повторно — это безопасно)
try:
    import nltk
    nltk.download("wordnet", quiet=True)
    nltk.download("omw-1.4", quiet=True)
    nltk.download("punkt", quiet=True)
    nltk.download("stopwords", quiet=True)
except Exception as e:
    print("NLTK download warning:", e)

from nltk.corpus import wordnet as wn
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

STOP = set(stopwords.words("english"))


## Данные (встроенные)

In [15]:
SENTENCES_EN = [
    "He deposited cash at the bank before it closed.",
    "The fishermen waited by the bank at dawn.",
    "The bass was hard to catch in the lake.",
    "His voice had a deep bass that filled the hall."
]

# GOLD для EN: целевые токены и проверка по глоссам (регексп по определению synset'а)
# Формат: (sent_id, target_token_index, lemma, pos, regex_on_gloss)
GOLD_EN = [
    (0, 5, "bank", "n", r"financial institution|depository financial institution"),
    (1, 5, "bank", "n", r"sloping|slope|embankment|river bank"),
    (2, 1, "bass", "n", r"fish|perch-like|marine fish|the lean-fleshed|freshwater"),  # рыба
    (3, 4, "bass", "n", r"lowest adult male|low musical|low pitch|bass part|lowest part in harmonized music"),  # музыка/голос
]

SENTENCES_ES = [
    "Depositó efectivo en el banco antes de que cerrara.",
    "Los pescadores esperaron junto a la orilla al amanecer."
]
SENTENCES_IT = [
    "Ha depositato contanti in banca prima che chiudesse.",
    "I pescatori hanno aspettato sulla riva all'alba."
]

TARGETS_ES = [
    (0, 3, "banco", "n"),
    (1, 5, "orilla", "n"),
]

TARGETS_IT = [
    (0, 3, "banca", "n"),
    (1, 5, "riva", "n"),
]


### ConceptNet-mini (встроенный TSV)

In [16]:

CONCEPTNET_MINI_TSV = """# src_lang	src_lemma	rel	tgt_lang	tgt_lemma	weight
en	bank	RelatedTo	en	money	1.7
en	bank	RelatedTo	en	river	1.5
en	bass	RelatedTo	en	music	1.8
en	bass	RelatedTo	en	fish	1.6
es	banco	RelatedTo	es	dinero	1.7
it	banca	RelatedTo	it	denaro	1.7
it	riva	RelatedTo	it	fiume	1.4
"""

def conceptnet_index_from_tsv(tsv_text: str) -> Dict[str, Dict[str, List[Tuple[str,str,float]]]]:
    
    idx: Dict[str, Dict[str, List[Tuple[str,str,float]]]] = {}
    
    for line in tsv_text.splitlines():
    
        if not line.strip() or line.startswith("#"): 
            continue
        
        parts = line.split("\t")
        if len(parts) != 6:
            continue
        
        src_lang, src_lemma, rel, tgt_lang, tgt_lemma, weight = parts
        idx.setdefault(src_lang, {}).setdefault(src_lemma, []).append((rel, tgt_lemma, float(weight)))
    return idx

CN_IDX = conceptnet_index_from_tsv(CONCEPTNET_MINI_TSV)
print("ConceptNet-mini edges for 'bank':", CN_IDX.get("en", {}).get("bank", []))


ConceptNet-mini edges for 'bank': [('RelatedTo', 'money', 1.7), ('RelatedTo', 'river', 1.5)]


## Утилиты

In [17]:

LANG_MAP = {
    "en": "eng",
    "es": "spa",
    "it": "ita",
}

def normalize_tokens(text: str) -> List[str]:
    toks = [t.lower() for t in word_tokenize(text)]
    
    # работаем с расширенной латиницей, сильно не запариваемся
    toks = [re.sub(r"[^a-záéíóúüñàèìòù]", "", t) for t in toks]
    toks = [t for t in toks if t and t not in STOP]
    return toks

def synset_gloss_text(ss, lang: str) -> str:
    gloss = ss.definition()
    try:
        ex = " ".join(ss.examples())
    except Exception:
        ex = ""
    return (gloss + " " + ex).strip()


## TODO-1: Кандидаты из WordNet/OMW

In [18]:

from nltk.corpus.reader.wordnet import Synset

def get_synsets(lemma: str, pos: str, lang: str) -> List[Synset]:
    """Вернуть список Synset для (lemma, pos, lang).
    Подсказки:
      - Для EN можно использовать wn.synsets(lemma, pos=pos)
      - Для ES/IT: отфильтровать synset'ы, где среди lemmas(lang_map) встречается наша лемма
    """
    if lang == "en":
        return wn.synsets(lemma, pos=pos)
    # TODO: реализуйте прочее
    raise NotImplementedError


## TODO-2/3: Счёты Lesk и ConceptNet + комбинация

In [19]:

def score_lesk(context_tokens: List[str], gloss_tokens: List[str]) -> float:
    """Упрощённый Lesk:
      score = |пересечение| / sqrt(len(gloss_tokens)+1)
    """
    # TODO
    raise NotImplementedError

def score_conceptnet(lemma: str, lang: str, context_tokens: List[str],
                     cn_index: Dict[str, Dict[str, List[Tuple[str,str,float]]]]) -> float:
    """Суммируем веса для соседей tgt_lemma, встречающихся в контексте.
       Нормируем делением на (1 + len(context_tokens)).
    """
    # TODO
    raise NotImplementedError

def combine_scores(lesk_score: float, cn_score: float, alpha: float = 0.7) -> float:
    """Итог: alpha * lesk + (1 - alpha) * cn"""
    # TODO
    raise NotImplementedError


## TODO-4: Разрешение неоднозначности для одного токена

In [20]:

def disambiguate_token(sentence: str, target_index: int, lemma: str, pos: str, lang: str,
                       alpha: float = 0.7) -> Optional[Tuple[Synset, float]]:
    
    toks = nltk.word_tokenize(sentence)
    ctx = normalize_tokens(" ".join(t for i,t in enumerate(toks) if i != target_index))
    candidates = get_synsets(lemma, pos, lang)
    
    if not candidates:
        return None
    
    best = None
    best_s = -1.0
    
    for ss in candidates:
        gloss_toks = normalize_tokens(synset_gloss_text(ss, lang))
        s_lesk = score_lesk(ctx, gloss_toks)
        s_cn = score_conceptnet(lemma, lang, ctx, CN_IDX)
        s = combine_scores(s_lesk, s_cn, alpha=alpha)
        if s > best_s:
            best = ss
            best_s = s
    return (best, best_s)


## TODO-5: Оценка на EN через регексп к глоссе

In [21]:

def evaluate_en(sentences: List[str], gold) -> Tuple[int,int,float]:
    correct = 0
    total = 0
    for (sid, tidx, lemma, pos, gloss_regex) in gold:
        out = disambiguate_token(sentences[sid], tidx, lemma, pos, "en")
        total += 1
        if out is None:
            continue
        ss, sc = out
        if re.search(gloss_regex, ss.definition(), re.I):
            correct += 1
    acc = correct/total if total else 0.0
    return correct, total, acc


## Тесты (ожидаемые ЧИСЛА)

In [22]:

# --- ТЕСТ 1 ---
try:
    c_bank_en = get_synsets("bank", "n", "en")
    print("EN/bank candidates:", len(c_bank_en))
    assert len(c_bank_en) >= 10, "Ожидаем >=10 кандидатов для 'bank.n' (EN)"
except NotImplementedError:
    print("Сначала реализуйте get_synsets.")

# --- ТЕСТ 2 ---
try:
    ctx = normalize_tokens("He deposited cash at the bank before it closed.")
    gloss_toks = normalize_tokens("a financial institution that accepts deposits")
    s_lesk = score_lesk(ctx, gloss_toks)
    print("Lesk(score) =", round(s_lesk, 4))
    assert s_lesk > 0.3, "Ожидаем Lesk > 0.3"
except NotImplementedError:
    pass

# --- ТЕСТ 3 ---
try:
    ctx2 = normalize_tokens("They sat by the river near the old bank.")
    s_cn = score_conceptnet("bank", "en", ctx2, CN_IDX)
    print("ConceptNet(score, river-context) =", round(s_cn, 4))
    assert s_cn >= 0.05, "Ожидаем положительный вклад ConceptNet"
except NotImplementedError:
    pass

# --- ТЕСТ 4 ---
try:
    ss_fin, sc_fin = disambiguate_token(SENTENCES_EN[0], 5, "bank", "n", "en")
    ss_riv, sc_riv = disambiguate_token(SENTENCES_EN[1], 5, "bank", "n", "en")
    print("Chosen gloss (finance):", ss_fin.definition())
    print("Chosen gloss (river):", ss_riv.definition())
    assert re.search(GOLD_EN[0][4], ss_fin.definition(), re.I), "Ожидаем финансовый 'bank'"
    assert re.search(GOLD_EN[1][4], ss_riv.definition(), re.I), "Ожидаем 'river bank'"
except NotImplementedError:
    pass

# --- ТЕСТ 5 ---
try:
    corr, total, acc = evaluate_en(SENTENCES_EN, GOLD_EN)
    print("EN accuracy:", corr, total, round(acc, 3))
    assert total == 4 and corr == 4 and abs(acc - 1.0) < 1e-9, "Ожидаем 4/4 = 1.0"
except NotImplementedError:
    pass

# --- ТЕСТ 6 ---
try:
    found_es = sum(1 for (sid, tidx, lemma, pos) in TARGETS_ES if get_synsets(lemma, pos, "es"))
    found_it = sum(1 for (sid, tidx, lemma, pos) in TARGETS_IT if get_synsets(lemma, pos, "it"))
    print("Coverage ES:", found_es, "/", len(TARGETS_ES))
    print("Coverage IT:", found_it, "/", len(TARGETS_IT))
    assert found_es == len(TARGETS_ES) and found_it == len(TARGETS_IT), "Ожидаем полное покрытие"
except NotImplementedError:
    pass


EN/bank candidates: 10
